# Assignment 4.2: Convolutional neural networks

You will build five convolutional networks in PyTorch and train each one to classify 28 by 28 pixel
grayscale images from two datasets: Fashion-MNIST (photos of clothing) and MNIST (handwritten
digits). The data loading, the training and testing function, the cross-validation, and the plots
are provided. Your job is to build the five architectures from the spec, choose how long to train
each one, and write the report.

Every model is evaluated with 5-fold cross-validation over a fixed sample of 20,000 images from each
dataset, 2,000 from each class. The sample is split into five folds of 4,000; the model is trained
five times, each time on four folds (16,000 images), and tested on the fold it did not see. The
results are the averages over the five runs.

Six code cells are yours: the five `build_model` functions and the settings cell. Everything else is
provided and already runs.

# Running in Colab

Open this notebook from its Colab link in the spec, then save your own copy right away with
**File > Save a copy in Drive**. Work in that copy, or your changes are lost when Colab closes.
Before you run anything, choose **Runtime > Change runtime type > T4 GPU**. This notebook trains
convolutional networks and is much slower without a GPU.

The next cell copies the course repository onto the Colab machine so the notebook finds its data in
`../data`. Run it first every time you open the notebook. On your own computer it does nothing.

In [ ]:
import os, sys, subprocess

# in colab, fetch the course repository so ../data exists
if "google.colab" in sys.modules:
    if not os.path.exists("/content/fund-ml-project-4"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/zacharytgray/fund-ml-project-4",
                        "/content/fund-ml-project-4"], check=True)
    os.chdir("/content/fund-ml-project-4/assignment")

# Import libraries
Do not use any other Python library.

numpy - Linear algebra library for handling vectors and matrices, collectively processed as numpy arrays.

matplotlib - Graphing library for visualizing results.

torch - PyTorch, the neural network library. `torch.nn` holds the layers and the loss function.

torchvision - PyTorch's image library, from which we source both datasets.

time - Simple library for timing code.

os, sys, subprocess - Used only by the Colab setup cell above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets
from time import time

# Device

Training runs on a GPU when PyTorch finds one: Colab's T4, CUDA on an NVIDIA card, or MPS on an Apple
silicon Mac. Otherwise it runs on the CPU, which works but is much slower. In Colab this cell should
print `cuda`.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using {device}")

# Load Datasets

`load_images` downloads one dataset into `../data/` the first time it runs and reads it from there
afterwards. Both datasets have ten classes and come split into 60,000 training images and 10,000
test images. Cross-validation makes its own splits, so `load_images` joins the two into one set of
70,000 images. It then keeps a fixed sample of `N_IMAGES` of them, the same number from each class.
Each image is one channel of 28 by 28 pixels with intensities from 0 to 1.

Inputs:
*   **name**: "fashion" or "digits" (string)

Outputs:
*   **X**: tensor of shape (20000, 1, 28, 28), the images
*   **y**: tensor of 20,000 integer class labels
*   **class_names**: list of the ten class names

# Function: make_folds

Shuffles the indices of the images with a fixed seed and cuts them into `N_FOLDS` folds of equal
size. It returns the five (training indices, test indices) pairs: each fold is the test set once,
and the other four are its training set.

In [ ]:
BATCH_SIZE = 64
N_FOLDS = 5
N_IMAGES = 20000
SEED = 0


def load_images(name, plot=True):
    source = datasets.FashionMNIST if name == "fashion" else datasets.MNIST
    train_data = source(root="../data", train=True, download=True)
    test_data = source(root="../data", train=False, download=True)
    # join the two splits; pixel values from 0-255 to 0-1, plus a channel dimension
    X = torch.cat([train_data.data, test_data.data]).float().div(255).unsqueeze(1)
    y = torch.cat([train_data.targets, test_data.targets])

    if plot:
        plt.figure(figsize=(10, 5))
        for i in range(10):
            plt.subplot(2, 5, i + 1)
            plt.imshow(X[i, 0], cmap="gray")
            plt.title(train_data.classes[int(y[i])])
            plt.axis("off")
        plt.tight_layout()
        plt.show()

    # keep N_IMAGES, the same number from each of the ten classes
    generator = torch.Generator().manual_seed(SEED)
    keep = []
    for c in range(10):
        index = torch.nonzero(y == c).flatten()
        keep.append(index[torch.randperm(len(index), generator=generator)[:N_IMAGES // 10]])
    keep = torch.cat(keep)
    X, y = X[keep], y[keep]

    return X.to(device), y.to(device), train_data.classes


def make_folds(n):
    order = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    folds = order.chunk(N_FOLDS)
    return [(torch.cat(folds[:k] + folds[k + 1:]), folds[k]) for k in range(N_FOLDS)]

# Models

Every model is an `nn.Sequential` stack built from these layers. Each one takes a batch of images of
shape (batch, 1, 28, 28) and returns ten scores per image, one per class.

*   [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html): `nn.Conv2d(in_channels, out_channels, kernel_size=3, padding="same")`. The number of filters is `out_channels`. With `padding="same"` the height and width do not change.
*   [ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html): `nn.ReLU()`.
*   [MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html): `nn.MaxPool2d(2)`. It halves the height and the width, rounding down.
*   [Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html): `nn.Dropout(0.5)`. During training it sets each value to 0 with probability 0.5.
*   [BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html): `nn.BatchNorm2d(channels)`. It normalizes each channel over the batch, then rescales it with two learned values per channel.
*   [Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html): `nn.Flatten()`. It turns each image's channels by height by width block into one vector.
*   [Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html): `nn.Linear(in_features, out_features)`. The first linear layer after `Flatten` takes channels times height times width inputs, so work out the image size after each pooling layer.

The last layer has ten outputs and no softmax. The cross-entropy loss applies the softmax itself.

# Function: build_model_1A

Two blocks of two convolutions. The first two `Conv2d` layers have 16 filters and the second two
have 32.

```
Conv2d(16) -> ReLU -> Conv2d(16) -> ReLU -> MaxPool2d(2)
Conv2d(32) -> ReLU -> Conv2d(32) -> ReLU -> MaxPool2d(2)
Flatten -> Linear(1024) -> ReLU -> Linear(10)
```

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_1A():
    # 1. first block: two 16-filter convolutions with relus, then max pooling
    # 2. second block: two 32-filter convolutions with relus, then max pooling
    # 3. flatten, a 1024-unit linear layer with relu, and the 10-class output layer
    # TODO
    pass

# Function: build_model_1B

The same layers as Model 1A with four times as many filters: 64 in the first block and 128 in the
second.

```
Conv2d(64) -> ReLU -> Conv2d(64) -> ReLU -> MaxPool2d(2)
Conv2d(128) -> ReLU -> Conv2d(128) -> ReLU -> MaxPool2d(2)
Flatten -> Linear(1024) -> ReLU -> Linear(10)
```

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_1B():
    # 1. first block: two 64-filter convolutions with relus, then max pooling
    # 2. second block: two 128-filter convolutions with relus, then max pooling
    # 3. flatten, a 1024-unit linear layer with relu, and the 10-class output layer
    # TODO
    pass

# Function: build_model_2

Model 1B with a dropout layer after each max pooling layer.

```
Conv2d(64) -> ReLU -> Conv2d(64) -> ReLU -> MaxPool2d(2) -> Dropout(0.5)
Conv2d(128) -> ReLU -> Conv2d(128) -> ReLU -> MaxPool2d(2) -> Dropout(0.5)
Flatten -> Linear(1024) -> ReLU -> Linear(10)
```

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_2():
    # 1. model 1B's first block, then dropout
    # 2. model 1B's second block, then dropout
    # 3. the same flatten and linear layers as model 1B
    # TODO
    pass

# Function: build_model_3

Model 1B with a batch normalization layer after the ReLU of each convolution.

```
Conv2d(64) -> ReLU -> BatchNorm2d -> Conv2d(64) -> ReLU -> BatchNorm2d -> MaxPool2d(2)
Conv2d(128) -> ReLU -> BatchNorm2d -> Conv2d(128) -> ReLU -> BatchNorm2d -> MaxPool2d(2)
Flatten -> Linear(1024) -> ReLU -> Linear(10)
```

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_3():
    # 1. model 1B's first block, with batch normalization after each convolution's relu
    # 2. model 1B's second block, with batch normalization after each convolution's relu
    # 3. the same flatten and linear layers as model 1B
    # TODO
    pass

# Function: build_model_4

A deeper network: three blocks of three convolutions, with 64, 128, and 256 filters, then two hidden
linear layers. Each convolution is followed by a ReLU and a batch normalization layer, and each block
ends with max pooling and dropout.

```
3 x [Conv2d(64) -> ReLU -> BatchNorm2d]  -> MaxPool2d(2) -> Dropout(0.5)
3 x [Conv2d(128) -> ReLU -> BatchNorm2d] -> MaxPool2d(2) -> Dropout(0.5)
3 x [Conv2d(256) -> ReLU -> BatchNorm2d] -> MaxPool2d(2) -> Dropout(0.5)
Flatten -> Linear(1024) -> ReLU -> Linear(512) -> ReLU -> Linear(10)
```

Output:
*   **model**: an `nn.Sequential` network

In [ ]:
def build_model_4():
    # 1. three blocks of three convolutions, each followed by relu and batch normalization
    # 2. 64, 128, and 256 filters in the three blocks; max pooling and dropout end each block
    # 3. flatten, linear layers of 1024 and 512 units with relus, and the 10-class output layer
    # TODO
    pass

# Settings

All models use the Adam optimizer with a learning rate of `LEARNING_RATE`. Model 4 is also trained a
second time with `MODEL_4_SECOND_RATE`, so you can see what the learning rate does to it.

Choose the number of training epochs of each model; each model uses the same number on both
datasets. Start every model with more epochs than you think it needs, then read its loss plot. Set
the epochs to the point of diminishing returns, where more training stops lowering the test loss.
Dropout makes training noisier, so Model 2 may need more epochs than Model 1B. Batch normalization
usually speeds up convergence, so Model 3 may need fewer.

In [ ]:
LEARNING_RATE = 0.001
MODEL_4_SECOND_RATE = 0.01

# 1. for each model, the number of epochs at the point of diminishing returns
# TODO
EPOCHS = {"1A": None, "1B": None, "2": None, "3": None, "4": None}

# Function: train_and_test

This function is provided. It trains one model with the Adam optimizer on the cross-entropy loss,
on minibatches of `BATCH_SIZE` training images in a new random order every epoch. After every epoch
it computes the loss and accuracy on the whole training set and on the whole test set, both with the
model in evaluation mode (dropout off, batch normalization using its running averages), so the two
are measured the same way. Those measurements never change the weights. It also times the training,
not counting the measurements.

Inputs:
*   **model**: a network from one of your build functions, already on `device`
*   **X**, **y**: the images and labels from `load_images`
*   **train_index**, **test_index**: the indices of the training and test images, from `make_folds`
*   **epochs**: number of passes over the training set (integer)
*   **learning_rate**: Adam learning rate (float)

Output:
*   **history**: dictionary with per-epoch lists **train_loss**, **train_accuracy**, **test_loss**, **test_accuracy**, and the total training time **seconds**

# Function: cross_validate

This function is provided. It builds a fresh model from the same seed for each fold, trains and tests
it with `train_and_test`, and returns the five histories.

# Function: count_parameters

This function is provided. It returns the number of trainable values in a network.

# Function: plot_history

This function is provided. It draws the training and test loss, and the training and test accuracy,
for every epoch of one model, averaged over the five folds. The test curve of each single fold is
drawn faintly behind.

In [ ]:
def train_and_test(model, X, y, train_index, test_index, epochs, learning_rate):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    generator = torch.Generator().manual_seed(SEED)
    history = {"train_loss": [], "train_accuracy": [], "test_loss": [], "test_accuracy": []}
    seconds = 0.0
    for epoch in range(epochs):
        start = time()
        model.train()
        order = train_index[torch.randperm(len(train_index), generator=generator)].to(device)
        for i in range(0, len(order), BATCH_SIZE):
            batch = order[i:i + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(model(X[batch]), y[batch])
            loss.backward()
            optimizer.step()
        if device == "cuda":
            torch.cuda.synchronize()
        seconds += time() - start

        for split, index in [("train", train_index), ("test", test_index)]:
            loss, accuracy = evaluate(model, X, y, index, loss_fn)
            history[f"{split}_loss"].append(loss)
            history[f"{split}_accuracy"].append(accuracy)
    history["seconds"] = seconds
    return history


def evaluate(model, X, y, index, loss_fn):
    # mean loss and accuracy over the given images, in evaluation mode
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(index), 1000):
            batch = index[i:i + 1000].to(device)
            scores = model(X[batch])
            total_loss += loss_fn(scores, y[batch]).item() * len(batch)
            correct += (scores.argmax(dim=1) == y[batch]).sum().item()
    return total_loss / len(index), correct / len(index) * 100


def cross_validate(build, X, y, folds, epochs, learning_rate):
    histories = []
    for k, (train_index, test_index) in enumerate(folds):
        torch.manual_seed(SEED)
        model = build().to(device)
        history = train_and_test(model, X, y, train_index, test_index, epochs, learning_rate)
        print(f"  fold {k + 1}: train acc {history['train_accuracy'][-1]:.2f}%  "
              f"test acc {history['test_accuracy'][-1]:.2f}%  {history['seconds']:.0f} s")
        histories.append(history)
    return histories


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def plot_history(histories, title):
    mean = {key: np.mean([h[key] for h in histories], axis=0)
            for key in ["train_loss", "test_loss", "train_accuracy", "test_accuracy"]}
    epochs = range(1, len(mean["train_loss"]) + 1)
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    for h in histories:
        ax_loss.plot(epochs, h["test_loss"], color="tab:orange", alpha=0.25, linewidth=0.8)
        ax_acc.plot(epochs, h["test_accuracy"], color="tab:orange", alpha=0.25, linewidth=0.8)
    ax_loss.plot(epochs, mean["train_loss"], color="tab:blue", label="train loss (mean of folds)")
    ax_loss.plot(epochs, mean["test_loss"], color="tab:orange", label="test loss (mean of folds)")
    ax_loss.set_xlabel("epoch")
    ax_loss.set_ylabel("cross-entropy loss")
    ax_loss.legend()
    ax_acc.plot(epochs, mean["train_accuracy"], color="tab:blue", label="train accuracy (mean of folds)")
    ax_acc.plot(epochs, mean["test_accuracy"], color="tab:orange", label="test accuracy (mean of folds)")
    ax_acc.set_xlabel("epoch")
    ax_acc.set_ylabel("accuracy (%)")
    ax_acc.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

# Main code

Loads each dataset and makes its five folds. Then, for each model, it prints the network, runs the
cross-validation, and plots the history averaged over the folds. Model 4 runs twice: at
`LEARNING_RATE` and at `MODEL_4_SECOND_RATE`. This is the slowest cell in the project, and its run time grows with the epochs you
choose. Run it once your build functions and settings are final.

In [ ]:
RUNS = [
    ("1A", build_model_1A, LEARNING_RATE),
    ("1B", build_model_1B, LEARNING_RATE),
    ("2", build_model_2, LEARNING_RATE),
    ("3", build_model_3, LEARNING_RATE),
    ("4", build_model_4, LEARNING_RATE),
    ("4", build_model_4, MODEL_4_SECOND_RATE),
]

results = []
for dataset_name in ["fashion", "digits"]:
    X, y, class_names = load_images(dataset_name)
    folds = make_folds(len(y))
    for model_name, build, learning_rate in RUNS:
        torch.manual_seed(SEED)
        model = build()
        print(f"\nModel {model_name} on {dataset_name}, learning rate {learning_rate}: "
              f"{count_parameters(model)} parameters")
        if learning_rate == LEARNING_RATE:
            print(model)
        histories = cross_validate(build, X, y, folds, EPOCHS[model_name], learning_rate)
        plot_history(histories, f"Model {model_name} on {dataset_name}, learning rate {learning_rate}")
        test_accuracies = [h["test_accuracy"][-1] for h in histories]
        results.append({
            "dataset": dataset_name, "model": model_name, "parameters": count_parameters(model),
            "epochs": EPOCHS[model_name], "learning_rate": learning_rate,
            "train_accuracy": np.mean([h["train_accuracy"][-1] for h in histories]),
            "test_accuracy": np.mean(test_accuracies), "test_std": np.std(test_accuracies),
            "test_loss": np.mean([h["test_loss"][-1] for h in histories]),
            "seconds": np.mean([h["seconds"] for h in histories]),
        })

# Results table

One row per model and dataset: parameter count, the settings it trained with, its training time
per fold, and its final training accuracy, test loss, and test accuracy, each averaged over the five
folds. `test std` is the standard deviation of the test accuracy across the folds.

In [ ]:
print(f"{'dataset':8} {'model':6} {'params':>9} {'epochs':>7} {'lr':>8} {'s/fold':>7} "
      f"{'train acc':>10} {'test loss':>10} {'test acc':>9} {'test std':>9}")
for r in results:
    print(f"{r['dataset']:8} {r['model']:6} {r['parameters']:9d} {r['epochs']:7d} {r['learning_rate']:8.4f} "
          f"{r['seconds']:7.1f} {r['train_accuracy']:9.2f}% {r['test_loss']:10.4f} {r['test_accuracy']:8.2f}% "
          f"{r['test_std']:8.2f}%")

# Report

Cover the following in your report.

*   Each model's loss and accuracy plots, the epochs you chose, its training time per fold, and its mean test accuracy over the five folds with its standard deviation. For Model 4, include both learning rates.
*   Compare Models 1A and 1B in accuracy and training time.
*   Compare Model 2 and Model 3 against Model 1B.
*   Compare Model 4 against the rest.
*   For Model 4, describe what changed at the second learning rate, in the loss curves and in the accuracy, and why.